# Manhattan plots for single-trait GWAS (summary-statistics based)

This notebook generates **Manhattan plots** from GWAS summary statistics to visualize genome-wide association signals for each VOC/trait. It is designed for **tetraploid potato GWAS outputs** (e.g., GEMMA/LMM) where results are stored as a table with per‑SNP p-values.

## What this notebook does
- **Load GWAS summary stats** from the configured table/path (trait, chromosome, position, p-value).
- **Clean and standardize columns** (e.g., ensure `chrom` and `start/position` are numeric/castable; drop missing p-values).
- **Compute −log10(p)** for plotting and (optionally) apply a significance threshold.
- **Create Manhattan plots**:
  - points across the genome colored by chromosome (or alternating chromosomes)
  - x-axis = genomic position (often “concatenated” across chromosomes)
  - y-axis = −log10(p)
  - horizontal line for genome-wide threshold (e.g., p = 1e-6)
- **Export figures** as PNGs (usually one plot per trait) into the configured output directory.

## Inputs (expected)
- GWAS table/dataframe with columns similar to:
  - `trait` (VOC name)
  - `chrom` (chromosome)
  - `start` or `position` (base-pair coordinate)
  - `p_wald` (p-value)
  -  marker ID (e.g., `rs`)
- Config variables such as `CONFIG_PATH`, output folder paths, and thresholds.

## Outputs
- Manhattan plot images (PNG) saved to the chosen output directory.
- finally the saved png plots will be used for manhatan_plot dashboard

In [0]:
import sys

MODULE_DIR = "/Volumes/bmqg/default_bronze/fatemeh/final_project/modules"

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

print("Module path added:", MODULE_DIR)


Module path added: /Volumes/bmqg/default_bronze/fatemeh/final_project/modules


In [0]:
import sys, importlib
sys.path.insert(0, MODULE_DIR)

import manhattan_module
importlib.reload(manhattan_module)


<module 'manhattan_module' from '/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/manhattan_module.py'>

In [0]:
import yaml

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)


gwas_table_newharvested = CONFIG["data"]["gwas_table_newharvested"]





In [0]:
import os
import re
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, trim
import yaml
from manhattan_module import (
    load_trait_gwas_subset,
    plot_manhattan
)


OUT_DIR = "/Volumes/bmqg/default_bronze/fatemeh/final_project/main_manhattan_plot_newharvested"
os.makedirs(OUT_DIR, exist_ok=True)



traits = (
    spark.table(gwas_table_newharvested)
    .select(trim(col("trait")).alias("trait"))
    .distinct()
    .toPandas()["trait"]
    .tolist()
)

print("N traits:", len(traits))


_show = plt.show
plt.show = lambda *args, **kwargs: None

# -----------------------------
# loop
# -----------------------------
for i, trait in enumerate(traits, 1):
    try:
        pdf = load_trait_gwas_subset(
            spark,
            trait,
            source_table=gwas_table_newharvested,  # ✅ درست
            p_sig=1e-6,
            random_fraction=0.05
        )

        if pdf.empty:
            print(f"Skipped empty: {trait}")
            continue

        plot_manhattan(pdf, trait, p_threshold=1e-6)

        fname = f"{i:02d}_{re.sub(r'[^\w\-.]+', '_', trait)}.png"

        plt.savefig(
            os.path.join(OUT_DIR, fname),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()
        print(f"Saved: {fname}")

    except Exception as e:
        print(f"Error {trait}: {e}")
        plt.close("all")

# restore show
plt.show = _show

print("Saved plots to:", OUT_DIR)

N traits: 38
Saved: 01_Decanal.png
Saved: 02_Butanoic_acid.png
Saved: 03_2-Methylpropanal.png
Saved: 04_Ethyl_hexanoate.png
Saved: 05_2-Propanone_1-methoxy-.png
Saved: 06_6-Methyl-5-hepten-2-one.png
Saved: 07_Hexanal.png
Saved: 08_Pentanal.png
Saved: 09_2_3-Pentanedione.png
Saved: 10_Benzoic_acid.png
Saved: 11_Acetic_acid_ethenyl_ester.png
Saved: 12_Furan_2-pentyl-.png
Saved: 13_Hexanoic_acid.png
Saved: 14_Hexanol.png
Saved: 15__E_-2-Decenal.png
Saved: 16_Benzaldehyde.png
Saved: 17_Butanal_3-methyl-.png
Saved: 18_Phenol.png
Saved: 19_Dimethyl_disulfide.png


/root/.ipykernel/1161919/command-2652685117965944-3271788587:52: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(


Saved: 20_1-Penten-3-one.png
Saved: 21_Tetradecane.png
Saved: 22_Dimethyl_trisulfide.png
Saved: 23_2_3-Butanedione.png
Saved: 24_Benzyl_alcohol.png
Saved: 25_Eucalyptol.png
Saved: 26_nonanoic_acid.png
Saved: 27_Tridecane.png
Saved: 28_2-Ethylfuran.png
Saved: 29_3-Carene.png


/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/manhattan_module.py:385: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


Saved: 30_Dimethyl_phthalate.png


/Volumes/bmqg/default_bronze/fatemeh/final_project/modules/manhattan_module.py:385: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/root/.ipykernel/1161919/command-2652685117965944-3271788587:52: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(


Saved: 31_1-Hexanol_2-ethyl-.png
Saved: 32_Acetic_acid_methyl_ester.png
Saved: 33_Linalool.png
Saved: 34_Nonanal.png
Saved: 35_3-Methylbutanal.png
Saved: 36_1-Octen-3-ol.png
Saved: 37_1-Pentanol.png
Saved: 38_1-Octyn-3-ol.png
Saved plots to: /Volumes/bmqg/default_bronze/fatemeh/final_project/main_manhattan_plot_newharvested
